# MacCrate fine-tuning story: evidence-first local replay

This notebook summarizes three Gemma 4 / LoRA experiments from retained local artifacts. It performs **no network calls, remote inference, training, or credential access**. Results are recomputed from JSON/JSONL evidence where available.

**Status discipline:** the codebook and support-router runs are completed historical results. The lexical-trigger experiment includes a completed 64-row main run. It reached 95.3% training, 96.9% dev, 89.1% locked test, and 68.8% challenge accuracy. It is reported as a near-miss against the user-revised 90% locked-test target, not rounded into a pass.

Run top-to-bottom with Python 3. `matplotlib` and `pandas` are optional; tables/plots degrade gracefully when unavailable.

## 1. Reproducibility contract and artifact map

Inputs are read-only and rooted beside this notebook:

- `../unsloth-codebook-hello-world/evaluation_report.json` — strict baseline/rescue/reload metrics.
- `../gemma4-support-router/formal_evaluation_report.json` — held-out, per-class, causality and persistence evidence.
- `protocol_manifest.json`, `candidate_selection.json` — frozen lexical protocol and baseline selection.
- `canary_gate_report.json`, `canary_completion_nothinking_gate_report.json`, `canary_checkpoint_summary.json` — canary causality and checkpoint evidence.
- `*_progress.jsonl` — optional local training traces.

Assertions below are executable claims, not decorative prose. Absolute remote adapter paths and likely secret fields are redacted before display.

In [ ]:
from pathlib import Path
import json, hashlib, re, math

HERE = Path.cwd().resolve()
ROOTS = {
    'codebook': HERE / 'evidence' / 'codebook',
    'router': HERE / 'evidence' / 'support-router',
    'lexical': HERE,
}
SECRET_KEY = re.compile(r'(token|secret|password|api[_-]?key|authorization|credential)', re.I)

def redact(obj, key=''):
    if SECRET_KEY.search(str(key)):
        return '<REDACTED>'
    if isinstance(obj, dict):
        return {k: redact(v, k) for k, v in obj.items()}
    if isinstance(obj, list):
        return [redact(v, key) for v in obj]
    if isinstance(obj, str) and (obj.startswith('/home/') or re.search(r'Bearer\s+\S+', obj, re.I)):
        return '<REDACTED_PATH>' if obj.startswith('/home/') else '<REDACTED>'
    return obj

def load_json(path, default=None):
    try:
        return json.loads(Path(path).read_text())
    except (FileNotFoundError, json.JSONDecodeError, OSError) as e:
        print(f'SKIP {path}: {type(e).__name__}')
        return {} if default is None else default

def load_jsonl(path):
    rows=[]
    try:
        for i, line in enumerate(Path(path).read_text().splitlines(), 1):
            if line.strip():
                try: rows.append(json.loads(line))
                except json.JSONDecodeError: print(f'SKIP malformed {path}:{i}')
    except OSError as e: print(f'SKIP {path}: {type(e).__name__}')
    return rows

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1<<20), b''): h.update(block)
    return h.hexdigest()

try:
    import pandas as pd
except ImportError:
    pd = None
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
print('Artifact roots:', {k: str(v) for k,v in ROOTS.items()})
print('Optional packages:', {'pandas': pd is not None, 'matplotlib': plt is not None})

In [ ]:
required = {
 'codebook': ['evaluation_report.json', 'FINAL_REPORT.md'],
 'router': ['formal_evaluation_report.json', 'FINAL_REPORT.md'],
 'lexical': ['protocol_manifest.json','candidate_selection.json','canary_gate_report.json',
             'canary_completion_nothinking_gate_report.json','canary_checkpoint_summary.json']
}
inventory=[]
for experiment, names in required.items():
    for name in names:
        p=ROOTS[experiment]/name
        inventory.append({'experiment':experiment,'file':name,'present':p.exists(),
                          'bytes':p.stat().st_size if p.exists() else None,
                          'sha256':sha256(p) if p.exists() else None})
if pd is not None:
    display(pd.DataFrame(inventory))
else:
    for row in inventory: print(row)
assert any(r['present'] for r in inventory), 'No experiment artifacts found'

## 2. Completed experiment A — synthetic codebook retrieval

Gate: baseline ≤5%, post-reload ≥90% (and ≥45/50), improvement ≥80 percentage points. The sole allowed rescue preserved leading-zero strings but still produced 0/50 exact matches. Character-position accuracy improved modestly; exact task success did not.

In [ ]:
codebook = load_json(ROOTS['codebook']/'evaluation_report.json')
cb_eval = codebook.get('evaluations', {})
cb_rows=[]
for name, m in cb_eval.items():
    cb_rows.append({'condition':name, 'exact_accuracy':m.get('exact_match_accuracy'),
                    'character_accuracy':m.get('character_position_accuracy'),
                    'format_rate':m.get('formatting_compliance_rate'), 'n':m.get('total')})
if pd is not None: display(pd.DataFrame(cb_rows))
else:
    for r in cb_rows: print(r)
if codebook:
    assert cb_eval['baseline']['exact_match_accuracy'] <= .05
    assert cb_eval['finetuned_after_reload']['exact_matches'] == 0
    assert codebook['criteria']['post_reload_ge_90pct'] is False
    assert codebook['pass'] is False
print('Completed result:', 'FAIL' if codebook and not codebook['pass'] else 'unavailable')

In [ ]:
if plt is not None and cb_rows:
    labels=[r['condition'].replace('finetuned_','FT\n') for r in cb_rows]
    x=range(len(labels))
    fig, ax=plt.subplots(figsize=(9,4))
    ax.bar([i-.18 for i in x],[100*(r['exact_accuracy'] or 0) for r in cb_rows],.36,label='Exact match')
    ax.bar([i+.18 for i in x],[100*(r['character_accuracy'] or 0) for r in cb_rows],.36,label='Character position')
    ax.axhline(90,color='crimson',ls='--',label='post-reload gate (90%)')
    ax.set(xticks=list(x),xticklabels=labels,ylabel='Accuracy (%)',title='Codebook: exact success stayed at zero')
    ax.legend(); plt.tight_layout(); plt.show()
else: print('Plot skipped: matplotlib or data unavailable')

## 3. Completed experiment B — three-way support router

Gate: ≥98% exact training diagnostic, ≥90% held-out after reload, and ≥90% for every class. The adapter caused a persistent +20.7 percentage-point held-out gain (33.3% → 54.0%), but SUMMIT remained at 0%, so the formal result is FAIL.

In [ ]:
router = load_json(ROOTS['router']/'formal_evaluation_report.json')
conditions=['baseline','training_diagnostic','adapter_off','adapter_on_before_reload','adapter_on_after_reload']
r_rows=[]
for c in conditions:
    m=router.get(c,{})
    r_rows.append({'condition':c,'accuracy':m.get('accuracy'),'valid_label_rate':m.get('valid_label_rate')})
if pd is not None: display(pd.DataFrame(r_rows))
else:
    for r in r_rows: print(r)
if router:
    assert math.isclose(router['adapter_on_after_reload']['accuracy'], .54)
    assert router['adapter_on_after_reload']['per_class']['SUMMIT']['correct'] == 0
    assert router['criteria']['adapter_improves_over_disabled'] is True
    assert router['persistence_prediction_match_rate'] == 1.0
    assert router['formal_result'] == 'FAIL'
print('Completed result:', router.get('formal_result','unavailable'))

In [ ]:
if plt is not None and router:
    pc=router['adapter_on_after_reload']['per_class']
    classes=list(pc); vals=[100*pc[k]['correct']/pc[k]['total'] for k in classes]
    fig,axs=plt.subplots(1,2,figsize=(10,4))
    axs[0].bar([r['condition'].replace('adapter_','') for r in r_rows if r['accuracy'] is not None],
               [100*r['accuracy'] for r in r_rows if r['accuracy'] is not None],color='#4C78A8')
    axs[0].axhline(90,color='crimson',ls='--'); axs[0].tick_params(axis='x',rotation=35)
    axs[0].set(title='Overall accuracy',ylabel='%')
    axs[1].bar(classes,vals,color=['#59A14F','#F28E2B','#E15759']); axs[1].axhline(90,color='crimson',ls='--')
    axs[1].set(title='Post-reload accuracy by class',ylabel='%')
    plt.tight_layout(); plt.show()
else: print('Plot skipped: matplotlib or data unavailable')

## 4. Lexical-trigger protocol — frozen design and baseline selection

A standalone lowercase `zorb` maps to COPPER; all negatives (including substrings and `z0rb`) map to SILVER. Candidate `c4` was preregistered by a deterministic suitability rule. Dataset counts and SHA-256 checksums are verified locally when files exist.

In [ ]:
protocol=load_json(ROOTS['lexical']/'protocol_manifest.json')
selection=load_json(ROOTS['lexical']/'candidate_selection.json')
print(redact({'model':protocol.get('model'),'trigger':protocol.get('trigger'),
              'labels':[protocol.get('positive_label'),protocol.get('negative_label')],
              'counts':protocol.get('counts'),'gates':protocol.get('gates'),
              'selected_candidate':selection.get('selected')}))
if protocol:
    assert protocol['counts']['canary_train.jsonl']==8
    assert protocol['gates']['canary_train_exact']=='8/8'
    for name, expected in protocol['checksums'].items():
        p=ROOTS['lexical']/name
        if p.exists(): assert sha256(p)==expected, f'checksum mismatch: {name}'
if selection:
    assert selection['selected']=='c4'
    assert next(x for x in selection['summaries'] if x['candidate_id']=='c4')['passes'] is True
print('Protocol and available dataset checksums: PASS')

In [ ]:
if plt is not None and selection:
    rows=selection['summaries']; labels=[r['candidate_id'] for r in rows]
    acc=[100*r['accuracy'] for r in rows]; dominance=[100*r['max_single_output_frequency'] for r in rows]
    x=range(len(rows)); fig,ax=plt.subplots(figsize=(7,4))
    ax.bar([i-.18 for i in x],acc,.36,label='Baseline accuracy')
    ax.bar([i+.18 for i in x],dominance,.36,label='Largest output share')
    ax.axhspan(25,75,color='green',alpha=.08,label='accuracy eligibility band')
    ax.axhline(80,color='crimson',ls='--',label='dominance ceiling')
    ax.set(xticks=list(x),xticklabels=labels,ylabel='Percent',title='Candidate baseline suitability'); ax.legend()
    plt.tight_layout(); plt.show()
else: print('Plot skipped: matplotlib or data unavailable')

## 5. Canary evidence — adapter causality improved, gate still failed

The original prompt setup showed no adapter effect (4/8). Completion-only/no-thinking inference reached 7/8, preserved both labels, and reproduced adapter-off behavior exactly. The original 8/8 scaling gate failed. The subsequent main run is retained as a separately disclosed user-authorized continuation, not evidence that the original gate passed. Checkpoints 10/20/30 show 3/8, 6/8, 7/8 respectively; none passes.

In [ ]:
canary=load_json(ROOTS['lexical']/'canary_gate_report.json')
canary_nt=load_json(ROOTS['lexical']/'canary_completion_nothinking_gate_report.json')
ckpt=load_json(ROOTS['lexical']/'canary_checkpoint_summary.json')
summary=[]
for name, report in [('original',canary),('completion_no_thinking',canary_nt)]:
    if report: summary.append({'evaluation':name,'off_before':report['off_before']['exact'],
                               'adapter_on':report['on']['exact'],'off_after':report['off_after']['exact'],
                               'total':report['on']['total'],'gate_pass':report['gate_pass']})
if pd is not None: display(pd.DataFrame(summary))
else:
    for r in summary: print(r)
if canary_nt:
    assert canary_nt['off_reproducible'] is True
    assert canary_nt['both_labels_correct'] is True
    assert canary_nt['on']['exact']==7 and canary_nt['on']['total']==8
    assert canary_nt['gate_pass'] is False
if ckpt: assert max(v['exact'] for v in ckpt.values()) < 8
print('Canary gate:', 'FAIL — 7/8; main continuation disclosed separately' if canary_nt and not canary_nt['gate_pass'] else 'unavailable')

In [ ]:
if plt is not None and (summary or ckpt):
    fig,axs=plt.subplots(1,2,figsize=(10,4))
    if summary:
        names=[r['evaluation'] for r in summary]
        axs[0].bar(names,[r['adapter_on'] for r in summary],color=['#9C755F','#59A14F'])
        axs[0].axhline(8,color='crimson',ls='--'); axs[0].set(title='Adapter-on canary exact',ylabel='Correct of 8',ylim=(0,8.5))
    if ckpt:
        steps=sorted((int(k),v['exact']) for k,v in ckpt.items())
        axs[1].plot([x for x,_ in steps],[y for _,y in steps],marker='o')
        axs[1].axhline(8,color='crimson',ls='--'); axs[1].set(title='Checkpoint selection',xlabel='Step',ylabel='Correct of 8',ylim=(0,8.5))
    plt.tight_layout(); plt.show()
else: print('Plot skipped: matplotlib or data unavailable')

## 6. Final main-run evidence

The main run is complete. The original strict gates and the later practical 90% target are both preserved. The locked test missed 90% by one example; challenge wording exposed substantial brittleness.


In [ ]:
main=load_json(ROOTS['lexical']/'main_evaluation_report.json')
dashboard=[
 {'experiment':'Codebook retrieval','execution':'completed','result':'FAIL','evidence':'evaluation_report.json'},
 {'experiment':'Support router','execution':'completed','result':'PARTIAL · 54%','evidence':'formal_evaluation_report.json'},
 {'experiment':'Lexical canary','execution':'completed','result':'7/8 · strict gate fail','evidence':'canary_completion_nothinking_gate_report.json'},
 {'experiment':'Lexical main','execution':'completed','result':'89.1% locked · near-miss','evidence':'main_evaluation_report.json'},
]
if pd is not None: display(pd.DataFrame(dashboard))
else:
    for row in dashboard: print(row)
if main:
    metrics={k:main[k]['accuracy'] for k in ['train_on','dev_on','test_on','challenge_on','test_after_reload']}
    print('Main accuracies:',metrics)
    assert main['train_on']['exact']==61 and main['train_on']['total']==64
    assert main['dev_on']['exact']==31 and main['dev_on']['total']==32
    assert main['test_on']['exact']==57 and main['test_on']['total']==64
    assert main['challenge_on']['exact']==22 and main['challenge_on']['total']==32
    assert main['off_before']['accuracy']==0.5
    assert main['off_reproducible'] is True and main['reload_reproducible'] is True


## 7. Interpretation — what counted as the win

1. **Mechanics and causality are demonstrated:** native-chat ingestion, real gradients, adapter off→on→off behavior, and exact reload persistence all worked.
2. **The main task nearly met the practical target:** locked accuracy rose from 50.0% adapter-off to 89.1% adapter-on; train and dev exceeded 90%. It remains one answer short of the revised 90% locked-test target.
3. **Challenge performance matters:** 68.8% on altered trigger contexts shows that the rule was learned but remained brittle outside the main distribution.
4. **The durable lesson:** low loss and a saved adapter are insufficient. Frozen data, exact scoring, causal adapter intervention, and reload tests make a fine-tuning claim credible.

This notebook intentionally contains no code that starts training, calls Studio, reads environment variables, or opens credential files.
